In [3]:
import jax
import jax.numpy as jnp
import numpy as np
from functools import partial

In [4]:
key_lhs = jax.random.PRNGKey(42)
key_rhs = jax.random.PRNGKey(43)

In [5]:
def autoencoder(z, r, neurons_pl, j=0, patience=50, error=jnp.inf, m=0, v=0, t=0, β_1=0.9, β_2=0.999, ϵ=1e-8, α=1e-3, guess=None):
    
    """
    neurons_pl must be a tuple
    """
    
    slice_positions_lhs, num_params_lhs = make_structure_lhs(neurons_pl)
    slice_positions_rhs, num_params_rhs = make_structure_rhs(neurons_pl)
    
    slice_positions_lhs = tuple(slice_positions_lhs)
    slice_positions_rhs = tuple(slice_positions_rhs)
    
    θ_lhs = 0.1 * jax.random.normal(key_lhs, shape=(num_params_lhs,))
    θ_rhs = 0.1 * jax.random.normal(key_rhs, shape=(num_params_rhs,))
    
    θ = jnp.concatenate([θ_lhs, θ_rhs], axis=0)
    
    while j < patience:
        
        θ, error_prime, m, v, t = adam_step(θ, z, r, m, v, t, neurons_pl, slice_positions_lhs, slice_positions_rhs, num_params_lhs)
        
        print(error_prime)
        
        if error_prime < error:
            
            j = 0
            error = error_prime
            θ_true = θ
        
        else:
            j += 1
        
    
    return θ_true

In [6]:
@partial(jax.jit, static_argnames=("neurons_pl", "slice_positions_lhs", "slice_positions_rhs", "num_params_lhs"))
def adam_step(θ, z, r, m, v, t, neurons_pl, slice_positions_lhs, slice_positions_rhs, num_params_lhs, β_1=0.9, β_2=0.999, ϵ=1e-8, α=1e-3):
    
    t += 1
        
    error_prime, g = jax.value_and_grad(loss_f)(θ, z, r, neurons_pl, slice_positions_lhs, slice_positions_rhs, num_params_lhs)
    
    m = β_1 * m + (1 - β_1) * g
    v = β_2 * v + jnp.multiply((1 - β_2) * g, g)
    m_hat = m / (1 - β_1**t)
    v_hat = v / (1 - β_2**t)
    
    θ = θ - jnp.true_divide(α * m_hat, jnp.sqrt(v_hat) + ϵ)
    
    return θ, error_prime, m, v, t

In [7]:
def loss_f(θ, z, r, neurons_pl, slice_positions_lhs, slice_positions_rhs, num_params_lhs):

    θ_lhs, θ_rhs = θ[:num_params_lhs], θ[num_params_lhs:]
    
    β = beta(z, neurons_pl, slice_positions_lhs, θ_lhs)
    
    x = make_characteristic_portfolios(z, r)
    f = factor(x, neurons_pl, slice_positions_rhs, θ_rhs)
    
    loss = jnp.mean((r - jnp.matvec(β, f))**2)
    
    return loss

In [8]:
def make_structure_lhs(neurons_pl):
    
    num_params = 0
    slice_positions = []
    
    for start, end in zip(neurons_pl[:-1], neurons_pl[1:]):
        slice_positions.append(num_params)
        num_params += start * end
        slice_positions.append(num_params)
        num_params += end
        
    slice_positions.append(num_params)
    
    return slice_positions, num_params

In [9]:
def make_structure_rhs(neurons_pl):
    
    num_params = 0
    slice_positions = []
    
    p, k = neurons_pl[0], neurons_pl[-1]
    
    slice_positions.append(num_params)
    num_params += p * k
    slice_positions.append(num_params)
    num_params += k
    slice_positions.append(num_params)
    
    return slice_positions, num_params

In [10]:
def make_characteristic_portfolios(z, r):
    
    x = jnp.linalg.solve(jnp.transpose(z, axes=(0,2,1)) @ z, jnp.matvec(jnp.transpose(z, axes=(0,2,1)), r)[...,None]).squeeze(-1)
    
    return x

In [11]:
def beta(z, neurons_pl, slice_positions, θ):

    β_prime = z
    
    for i, l in enumerate(slice_positions[:-3]):
        
        if i%2 != 0:
            continue
        
        layer_params_w = θ[slice_positions[i]:slice_positions[i+1]]
        layer_params_b = θ[slice_positions[i+1]:slice_positions[i+2]]
        w = layer_params_w.reshape(neurons_pl[i//2],-1)
        b = layer_params_b
        
        β = jnp.maximum(0.0, β_prime @ w + b)
        β_prime = β

    layer_params_w = θ[slice_positions[-3]:slice_positions[-2]]
    layer_params_b = θ[slice_positions[-2]:slice_positions[-1]]
    w = layer_params_w.reshape(neurons_pl[-2],-1)
    b = layer_params_b
    
    β = β_prime @ w + b
    
    return β

In [12]:
def factor(x, neurons_pl, slice_positions_rhs, θ_rhs):
    
    k = neurons_pl[-1]
    layer_params_w = θ_rhs[slice_positions_rhs[0]:slice_positions_rhs[1]]
    layer_params_b = θ_rhs[slice_positions_rhs[1]:slice_positions_rhs[2]]
    w = layer_params_w.reshape(k, -1)
    b = layer_params_b
    
    f = x @ w.T + b
    
    return f

In [13]:
key_1 = jax.random.PRNGKey(1)
key_2 = jax.random.PRNGKey(2)
key_3 = jax.random.PRNGKey(3)
key_4 = jax.random.PRNGKey(4)
key_5 = jax.random.PRNGKey(5)

N = 500
T = 800
Z_num = 50

Z = np.zeros((T,N,Z_num))

σ_char = jax.random.uniform(key_1, shape=(Z_num,), minval=0.0, maxval=10.0)
ρ_char = jax.random.uniform(key_2, shape=(Z_num,), minval=0.8, maxval=0.99)

w = jax.random.normal(key_3, shape=(T,N,Z_num,)) * σ_char

Z[0] = w[0]
for i in range(1, T):
    Z[i] = Z[i-1] * ρ_char + w[i]

Z = Z[500:,:,:]
T = 800-500

Z = 2 * Z.argsort(axis=1).argsort(axis=1) / (Z.shape[1] - 1) - 1

β = np.zeros((T, N, 3))

β[:,:,0] = np.tanh(Z[:,:,0])
β[:,:,1] = Z[:,:,1] * Z[:,:,2]
β[:,:,2] = np.abs(Z[:,:,3])


f = jax.random.normal(key_4, shape=(T, 3,))

μ_f = jnp.array([0.005, 0.010, 0.015])
σ_f = jnp.array([0.03, 0.04, 0.05])

f = f * σ_f + μ_f

σ_ν = 0.00

r = jnp.matvec(β, f) + σ_ν * jax.random.normal(key_5, shape=(T, N,))

In [32]:
target_params = autoencoder(Z, r, (50, 32, 3))

0.0020167313
0.0019174876
0.0018287667
0.0017500012
0.0016805589
0.0016197624
0.0015668999
0.0015212471
0.0014820795
0.0014486851
0.0014203786
0.0013965112
0.0013764787
0.001359723
0.0013457418
0.0013340851
0.0013243526
0.0013161941
0.0013093081
0.0013034395
0.0012983745
0.0012939406
0.0012899984
0.0012864374
0.0012831739
0.0012801453
0.0012773059
0.0012746276
0.0012720913
0.0012696855
0.0012674064
0.0012652541
0.0012632284
0.0012613295
0.0012595576
0.0012579121
0.0012563916
0.0012549922
0.0012537095
0.0012525362
0.0012514654
0.0012504879
0.0012495943
0.0012487759
0.0012480228
0.0012473252
0.0012466738
0.0012460597
0.0012454756
0.0012449146
0.0012443711
0.0012438399
0.0012433168
0.0012427993
0.0012422849
0.0012417722
0.00124126
0.0012407479
0.0012402352
0.0012397228
0.0012392114
0.0012387012
0.0012381928
0.0012376865
0.0012371831
0.001236683
0.0012361857
0.0012356917
0.0012352006
0.001234712
0.0012342255
0.0012337406
0.0012332577
0.0012327756
0.0012322941
0.0012318123
0.0012313297
0.00

0.00054270105
0.0005424345
0.00054216845
0.0005419024
0.00054163625
0.00054137036
0.00054110464
0.00054083916
0.0005405738
0.0005403084
0.0005400432
0.00053977786
0.0005395126
0.0005392475
0.0005389826
0.00053871784
0.00053845305
0.00053818827
0.0005379237
0.0005376591
0.0005373947
0.0005371304
0.0005368659
0.0005366012
0.00053633645
0.0005360716
0.0005358066
0.0005355415
0.0005352763
0.00053501106
0.00053474563
0.00053447986
0.00053421373
0.0005339473
0.00053368043
0.0005334129
0.00053314504
0.0005328767
0.000532608
0.00053233927
0.00053207064
0.0005318018
0.0005315327
0.0005312633
0.00053099357
0.0005307233
0.00053045293
0.0005301825
0.00052991166
0.0005296403
0.0005293685
0.00052909635
0.00052882393
0.00052855123
0.0005282776
0.00052800355
0.0005277293
0.00052745454
0.0005271793
0.0005269035
0.0005266278
0.0005263516
0.0005260751
0.00052579824
0.0005255211
0.0005252438
0.0005249662
0.00052468816
0.0005244097
0.00052413106
0.0005238521
0.0005235728
0.00052329304
0.0005230125
0.000522

0.00043239206
0.00043235804
0.000432324
0.00043229005
0.0004322562
0.0004322225
0.00043218883
0.0004321553
0.00043212177
0.00043208836
0.00043205507
0.00043202183
0.00043198874
0.00043195568
0.0004319227
0.00043188984
0.00043185704
0.00043182433
0.00043179168
0.00043175914
0.0004317267
0.0004316943
0.00043166193
0.00043162963
0.00043159738
0.0004315651
0.000431533
0.0004315009
0.0004314689
0.00043143696
0.00043140512
0.00043137328
0.00043134156
0.00043130992
0.00043127828
0.00043124665
0.0004312151
0.00043118358
0.0004311521
0.0004311206
0.00043108923
0.00043105785
0.0004310265
0.0004309952
0.00043096385
0.00043093256
0.00043090133
0.0004308701
0.00043083893
0.00043080776
0.00043077656
0.00043074533
0.00043071422
0.00043068308
0.00043065197
0.00043062083
0.0004305897
0.00043055858
0.0004305275
0.00043049632
0.00043046524
0.00043043407
0.00043040287
0.0004303717
0.00043034053
0.00043030936
0.00043027822
0.00043024708
0.00043021585
0.00043018474
0.0004301536
0.00043012248
0.00043009134
0

0.00037579896
0.00037564768
0.0003754969
0.00037534657
0.0003751967
0.00037504733
0.00037489846
0.0003747501
0.00037460215
0.00037445474
0.0003743079
0.00037416158
0.00037401568
0.0003738704
0.00037372563
0.0003735814
0.0003734376
0.00037329446
0.0003731519
0.00037301012
0.00037286922
0.00037272956
0.00037259157
0.0003724561
0.0003723248
0.0003722006
0.00037208846
0.00037199658
0.00037193618
0.00037191808
0.00037193732
0.0003719435
0.00037182856
0.0003715124
0.00037109497
0.0003708091
0.0003707528
0.00037078568
0.0003706982
0.00037043833
0.0003701666
0.0003700402
0.0003700128
0.00036992467
0.00036972147
0.0003695106
0.0003693941
0.0003693307
0.00036921687
0.0003690374
0.0003688718
0.0003687674
0.00036867938
0.0003685498
0.00036839047
0.00036825382
0.00036815196
0.00036804713
0.00036791398
0.0003677724
0.0003676515
0.0003675469
0.00036743341
0.00036730393
0.00036717582
0.00036706214
0.00036695454
0.00036683877
0.00036671534
0.00036659607
0.0003664856
0.00036637654
0.0003662619
0.0003661

0.00034491744
0.00034489812
0.00034488508
0.0003448775
0.0003448678
0.00034485265
0.00034483633
0.00034482384
0.00034481447
0.00034480388
0.00034479008
0.0003447756
0.00034476325
0.0003447528
0.0003447419
0.00034472902
0.00034471558
0.00034470332
0.0003446923
0.0003446811
0.00034466886
0.00034465618
0.00034464395
0.00034463263
0.00034462137
0.0003446096
0.0003445974
0.0003445854
0.0003445738
0.00034456252
0.000344551
0.00034453912
0.00034452727
0.0003445157
0.00034450428
0.00034449296
0.00034448144
0.0003444698
0.0003444583
0.000344447
0.00034443568
0.0003444244
0.00034441298
0.00034440155
0.0003443903
0.0003443791
0.000344368
0.00034435684
0.0003443456
0.00034433452
0.00034432343
0.0003443124
0.00034430134
0.00034429025
0.00034427922
0.00034426822
0.00034425728
0.00034424636
0.00034423545
0.00034422457
0.00034421377
0.00034420294
0.00034419217
0.00034418146
0.00034417075
0.00034416
0.00034414936
0.00034413874
0.00034412817
0.00034411755
0.00034410701
0.00034409654
0.000344086
0.000344

...

0.00033613318
0.00033613306
0.0003361331
0.000336133
0.000336133
0.0003361329
0.0003361329
0.0003361329
0.00033613286
0.00033613289
0.0003361328
0.00033613289
0.00033613286
0.00033613286
0.00033613286
0.00033613303
0.00033613318
0.00033613347
0.00033613402
0.00033613504
0.00033613658
0.0003361391
0.00033614342
0.00033615055
0.00033616245
0.00033618216
0.00033621467
0.0003362662
0.00033634607
0.00033645733
0.00033659438
0.00033671054
0.00033673935
0.00033660812
0.00033636988
0.00033617264
0.00033614304
0.00033625655
0.0003363771
0.0003363849
0.00033627302
0.0003361566
0.00033613938
0.00033620954
0.0003362704
0.00033625125
0.000336179
0.000336133
0.00033615174
0.00033619697
0.000336209
0.00033617593
0.0003361384
0.00033613516
0.0003361599
0.0003361766
0.0003361653
0.0003361412
0.00033613155
0.00033614217
0.00033615564
0.0003361546
0.00033614138
0.00033613166
0.0003361344
0.00033614293
0.00033614587
0.00033614
0.00033613277
0.00033613152
0.00033613594
0.00033613946
0.00033613795
0.0003361

0.00033610497
0.00033610468
0.0003361046
0.00033610457
0.0003361044
0.0003361042
0.0003361041
0.00033610396
0.00033610378
0.00033610367
0.00033610355
0.00033610352
0.00033610337
0.00033610337
0.0003361032
0.0003361031
0.00033610297
0.00033610288
0.00033610276
0.00033610268
0.0003361026
0.00033610244
0.00033610236
0.00033610233
0.00033610215
0.00033610215
0.00033610204
0.00033610186
0.00033610174
0.00033610177
0.0003361016
0.00033610157
0.00033610154
0.00033610137
0.00033610125
0.00033610122
0.00033610107
0.00033610102
0.00033610093
0.0003361009
0.00033610075
0.00033610073
0.0003361006
0.00033610055
0.00033610046
0.00033610046
0.00033610032
0.00033610014
0.00033610006
0.00033609994
0.00033609977
0.00033609965
0.00033609945
0.00033609936
0.00033609918
0.00033609907
0.00033609883
0.00033609895
0.00033609875
0.00033609857
0.00033609846
0.00033609834
0.00033609837
0.00033609816
0.00033609816
0.0003360981
0.00033609796
0.00033609782
0.00033609773
0.0003360977
0.00033609752
0.00033609747
0.00

0.00033608393
0.0003360649
0.0003360766
0.0003361001
0.00033611056
0.00033609895
0.00033607753
0.00033606516
0.00033606906
0.00033608155
0.0003360887
0.00033608422
0.00033607305
0.0003360652
0.00033606603
0.0003360722
0.00033607698
0.00033607575
0.00033607025
0.00033606528
0.00033606455
0.00033606743
0.00033607046
0.00033607072
0.00033606822
0.0003360653
0.00033606414
0.0003360651
0.0003360669
0.0003360677
0.00033606688
0.0003360653
0.00033606417
0.0003360642
0.00033606496
0.00033606563
0.00033606577
0.0003360651
0.00033606435
0.00033606385
0.0003360642
0.00033606452
0.00033606487
0.0003360647
0.0003360643
0.00033606394
0.00033606368
0.00033606382
0.00033606408
0.00033606414
0.00033606408
0.00033606388
0.0003360637
0.00033606365
0.00033606365
0.00033606365
0.0003360637
0.00033606376
0.00033606356
0.00033606347
0.0003360633
0.00033606344
0.00033606344
0.0003360634
0.0003360634
0.00033606336
0.00033606333
0.0003360632
0.0003360633
0.00033606327
0.00033606324
0.00033606312
0.00033606315
0

0.00033604624
0.00033608585
0.00033615108
0.0003361897
0.00033617605
0.00033612052
0.00033606513
0.00033604514
0.00033606432
0.00033609793
0.0003361148
0.0003361029
0.00033607296
0.0003360491
0.0003360461
0.00033606048
0.0003360761
0.00033607893
0.00033606787
0.0003360525
0.00033604447
0.00033604726
0.0003360559
0.0003360617
0.00033605992
0.0003360527
0.0003360458
0.00033604383
0.0003360468
0.00033605128
0.00033605282
0.00033605078
0.00033604694
0.000336044
0.00033604374
0.0003360455
0.00033604747
0.0003360479
0.0003360467
0.00033604476
0.0003360434
0.0003360434
0.0003360442
0.00033604514
0.00033604534
0.00033604476
0.00033604377
0.00033604313
0.00033604293
0.00033604325
0.0003360437
0.00033604377
0.00033604354
0.00033604304
0.00033604252
0.00033604223
0.0003360423
0.00033604237
0.0003360425
0.0003360424
0.00033604223
0.00033604188
0.00033604167
0.0003360415
0.00033604156
0.0003360415
0.00033604162
0.00033604162
0.00033604144
0.00033604135
0.00033604115
0.00033604106
0.00033604106
0.00

0.00033603061
0.0003360306
0.00033603053
0.00033603056
0.0003360306
0.00033603061
0.00033603061
0.00033603047
0.00033603047
0.00033603053
0.00033603053
0.00033603053
0.00033603056
0.00033603053
0.00033603047
0.0003360306
0.00033603053
0.00033603061
0.00033603053
0.00033603047
0.00033603053
0.00033603056
0.0003360305
0.00033603053
0.00033603061
0.0003360306
0.0003360304
0.00033603044
0.00033603044
0.0003360304
0.0003360304
0.0003360304
0.00033603032
0.00033603038
0.00033603038
0.00033603032
0.00033603032
0.00033603027
0.00033603027
0.00033603024
0.00033603027
0.00033603032
0.00033603032
0.00033603024
0.00033603032
0.0003360303
0.00033603032
0.00033603027
0.00033603027
0.00033603047
0.0003360304
0.00033603053
0.00033603073
0.00033603108
0.00033603163
0.00033603256
0.00033603405
0.00033603673
0.00033604106
0.00033604843
0.00033606053
0.00033608088
0.00033611403
0.0003361674
0.00033624793
0.00033636176
0.00033649275
0.00033660277
0.0003366052
0.00033646126
0.0003362227
0.0003360524
0.00033